# TinyDoc-VLM 768 Retrain (Kaggle)

Fully resumable full-model fine-tune on Kaggle GPU (free 30h/week).

Checkpoints + data live in private Hugging Face repos, so every Kaggle
session resumes from the last saved step. Add the **HF_TOKEN** secret:
left panel -> Add-ons -> Secrets.

Tune training with `--steps` etc. below. ~9.4 s/step on T4; use the **T4** (single) accelerator - batch 2 + grad-accum 4 fits and burns half the GPU quota of T4x2.

In [ ]:
import subprocess, sys, os

REPO_URL = 'https://github.com/eulogik/TinyDoc-VLM'
REPO = '/kaggle/working/tinydoc-vlm'

STEPS = os.environ.get('STEPS', '8000')
# SAFE default: batch 2 + grad-accum 4 = effective batch 8. Do not raise BATCH
# on the 16 GiB T4 - 5-tile images OOM the backward pass at batch 4+.
BATCH = os.environ.get('BATCH', '2')

if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        raise RuntimeError('HF_TOKEN not set. Add it to Kaggle Secrets (Settings > Secrets).')

# Always fresh clone: Kaggle kernels keep /kaggle/working between re-runs
# in the same session, so a stale clone would miss self-healing/torch-pin
# fixes. depth-1 clone ~10s; kaggle_train.py re-clones anyway as backup.
if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO])

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])

p = subprocess.run(
    [sys.executable, 'training/kaggle_train.py',
     '--steps', STEPS, '--batch-size', BATCH,
     # effective batch 8 (2x4). Synthetic 800x1050 pages tile into up to 5
     # tiles; batch 8 single-GPU OOMs the 16 GiB T4 in the backward pass.
     '--grad-accum', '4'],
    cwd=REPO,
)
sys.exit(p.returncode)